##### ***哈希表***
###### 哈希表的主要操作有：哈希表类的定义、hash函数、添加/修改、查找、删除、判断key是否存在、获取数量

In [3]:
# 哈希表类的定义
"""class HashTable:
    def __init__(self, capacity=10):
        self.capacity = capacity
        self.table = [None] * capacity

    # 哈希函数
    def _hash(self, key):
        return key % self.capacity
    
    # 添加键值对
    def put(self, key, value):
        index = self._hash(key)
        self.table[index] = (key, value)
"""

'class HashTable:\n    def __init__(self, capacity=10):\n        self.capacity = capacity\n        self.table = [None] * capacity\n\n    # 哈希函数\n    def _hash(self, key):\n        return key % self.capacity\n    \n    # 添加键值对\n    def put(self, key, value):\n        index = self._hash(key)\n        self.table[index] = (key, value)\n'

###### 由于之前学过简单的哈希函数很容易造成哈希冲突，所以接下来分别实现两种处理方法：链地址法和开放寻址法。

In [4]:
# 利用链地址法实现哈希表
class HashTable:
    def __init__(self, capacity=10):
        self.capacity = capacity
        self.table = [[] for _ in range(capacity)]

    # 哈希函数
    def _hash(self, key):
        return key % self.capacity

    # 添加/修改键值对
    def put(self, key, value):
        index = self._hash(key)
        bucket = self.table[index]
        for i in range(len(bucket)):
            if bucket[i][0] == key:
                bucket[i] = (key, value)
                return True
        bucket.append((key, value))
        return True

    # 获取键值对
    def get(self, key):
        index = self._hash(key)
        bucket = self.table[index]
        for i in range(len(bucket)):
            if bucket[i][0] == key:
                return bucket[i][1]
        return None

    # 删除
    def delete(self, key):
        index = self._hash(key)
        bucket = self.table[index]
        for i in range(len(bucket)):
            if bucket[i][0] == key:
                bucket.pop(i)
                return True
        return False

    # 判断元素是否存在
    def contains(self, key):
        index = self._hash(key)
        bucket = self.table[index]
        for i in range(len(bucket)):
            if bucket[i][0] == key:  
                return True
        return False    


    # 获取哈希表内的键值对数量
    def size(self):
        counter = 0
        for i in range(len(self.table)):
            bucket = self.table[i]
            counter += len(bucket)
        return counter

# =========================
# 哈希表测试
# =========================

hash_table = HashTable()

print("=== 初始状态 ===")
print(hash_table.table)
print(hash_table.size())
print(hash_table.contains(23))
print(hash_table.get(23))

print("=== 添加键值对 ===")
hash_table.put(23, "Alice")
hash_table.put(33, "Bob")
hash_table.put(43, "Carol")

print(hash_table.table)
print(hash_table.size())

print("=== 哈希冲突测试 ===")
print(hash_table._hash(23))
print(hash_table._hash(33))
print(hash_table._hash(43))
print(hash_table.table[3])

print("=== 获取键值对 ===")
print(hash_table.get(23))
print(hash_table.get(33))
print(hash_table.get(43))
print(hash_table.get(99))

print("=== 判断是否存在 ===")
print(hash_table.contains(23))
print(hash_table.contains(43))
print(hash_table.contains(99))

print("=== 重复 key 更新 ===")
hash_table.put(23, "David")
print(hash_table.get(23))
print(hash_table.table[3])
print(hash_table.size())

print("=== 删除键值对 ===")
print(hash_table.delete(33))
print(hash_table.table[3])
print(hash_table.size())

print("=== 删除不存在的 key ===")
print(hash_table.delete(99))
print(hash_table.size())

=== 初始状态 ===
[[], [], [], [], [], [], [], [], [], []]
0
False
None
=== 添加键值对 ===
[[], [], [], [(23, 'Alice'), (33, 'Bob'), (43, 'Carol')], [], [], [], [], [], []]
3
=== 哈希冲突测试 ===
3
3
3
[(23, 'Alice'), (33, 'Bob'), (43, 'Carol')]
=== 获取键值对 ===
Alice
Bob
Carol
None
=== 判断是否存在 ===
True
True
False
=== 重复 key 更新 ===
David
[(23, 'David'), (33, 'Bob'), (43, 'Carol')]
3
=== 删除键值对 ===
True
[(23, 'David'), (43, 'Carol')]
2
=== 删除不存在的 key ===
False
2


In [5]:
# DELETED 不能放在delete()里临时创建，否则每调用一次delete()都会生成新的对象
DELETED = object()
# 开放地址法
class OpenAddressHashTable:
    def __init__(self, capacity=10):
        self.capacity = capacity
        self.table = [None] * capacity

    def _hash(self, key):
        return key % self.capacity

    # 添加或更新键值对
    # 使用开放地址法中的线性探测解决哈希冲突
    def put(self, key, value):
        # 根据 key 计算初始哈希位置
        index = self._hash(key)

        # 记录探测过程中遇到的第一个墓碑位置
        # 墓碑表示该位置曾经存储过数据，但数据已经被删除
        first_deleted = None

        # 最多探测 capacity 次，避免哈希表已满时出现死循环
        for _ in range(self.capacity):

            # 情况1：当前位置从未被使用过
            if self.table[index] is None:

                # 如果之前遇到过墓碑，则优先复用第一个墓碑位置
                if first_deleted is not None:
                    self.table[first_deleted] = (key, value)

                # 如果没有遇到墓碑，则直接使用当前空位置
                else:
                    self.table[index] = (key, value)

                return True

            # 情况2：当前位置是墓碑
            if self.table[index] is DELETED:

                # 只记录遇到的第一个墓碑位置
                if first_deleted is None:
                    first_deleted = index

                # 不能立即插入，因为后面可能存在相同的 key
                # 因此继续向后进行线性探测
                index = (index + 1) % self.capacity
                continue

            # 情况3：当前位置存储的是正常键值对
            # 如果 key 相同，说明该 key 已经存在，直接更新 value
            if self.table[index][0] == key:
                self.table[index] = (key, value)
                return True

            # 情况4：当前位置被其他 key 占用，发生哈希冲突
            # 向后移动一个位置继续进行线性探测
            # 取模保证到达数组末尾后能够重新从下标 0 开始
            index = (index + 1) % self.capacity

        # 如果已经探测完整个哈希表，没有遇到 None
        # 但之前存在墓碑，则仍然可以复用该墓碑
        if first_deleted is not None:
            self.table[first_deleted] = (key, value)
            return True

        # 整个哈希表既没有空位置，也没有墓碑，插入失败
        return False

    def get(self, key):
        index = self._hash(key)
        for _ in range(self.capacity):
            if self.table[index] is None:
                return None
            if self.table[index] is DELETED:
                index = (index + 1) % self.capacity
                continue
            if self.table[index][0] == key:
                return self.table[index][1]
            index = (index + 1) % self.capacity
        return None

    def delete(self, key):
        # 删除操作与之前不同，不能单纯的直接将值设为None，因为这样后，接下来使用get操作会遇到None导致直接返回，所以需要给删除的地方打上特殊标记
        index = self._hash(key)
        for _ in range(self.capacity):
            if self.table[index] is None:
                return False
            if self.table[index] is DELETED:
                index = (index + 1) % self.capacity
                continue
            if self.table[index][0] == key:
                self.table[index] = DELETED
                return True
            index = (index + 1) % self.capacity
        return False

# =========================
# 开放地址法哈希表测试
# =========================

hash_table = OpenAddressHashTable()

print("=== 初始状态 ===")
print(hash_table.table)

print("\n=== 普通插入 ===")
print(hash_table.put(23, "Alice"))
print(hash_table.put(33, "Bob"))
print(hash_table.put(43, "Carol"))
print(hash_table.table)

print("\n=== 哈希冲突测试 ===")
print(hash_table._hash(23))
print(hash_table._hash(33))
print(hash_table._hash(43))
print(hash_table.table)

print("\n=== get 测试 ===")
print(hash_table.get(23))
print(hash_table.get(33))
print(hash_table.get(43))
print(hash_table.get(99))

print("\n=== 重复 key 更新 ===")
print(hash_table.put(23, "David"))
print(hash_table.get(23))
print(hash_table.table)

print("\n=== 删除测试 ===")
print(hash_table.delete(33))
print(hash_table.table)

print("\n=== 删除后继续查找后续元素 ===")
print(hash_table.get(43))

print("\n=== 删除不存在的 key ===")
print(hash_table.delete(99))

print("\n=== 墓碑复用测试 ===")
print(hash_table.put(53, "Eve"))
print(hash_table.table)
print(hash_table.get(53))

print("\n=== 环绕探测测试 ===")
wrap_table = OpenAddressHashTable(capacity=5)

print(wrap_table.put(4, "A"))
print(wrap_table.put(9, "B"))
print(wrap_table.put(14, "C"))
print(wrap_table.table)

print(wrap_table.get(4))
print(wrap_table.get(9))
print(wrap_table.get(14))

print("\n=== 哈希表填满测试 ===")
full_table = OpenAddressHashTable(capacity=3)

print(full_table.put(0, "A"))
print(full_table.put(1, "B"))
print(full_table.put(2, "C"))
print(full_table.table)

print("继续插入：")
print(full_table.put(3, "D"))
print(full_table.table)

print("\n=== 满表存在墓碑时重新插入 ===")
print(full_table.delete(1))
print(full_table.table)

print(full_table.put(4, "E"))
print(full_table.table)
print(full_table.get(4))

=== 初始状态 ===
[None, None, None, None, None, None, None, None, None, None]

=== 普通插入 ===
True
True
True
[None, None, None, (23, 'Alice'), (33, 'Bob'), (43, 'Carol'), None, None, None, None]

=== 哈希冲突测试 ===
3
3
3
[None, None, None, (23, 'Alice'), (33, 'Bob'), (43, 'Carol'), None, None, None, None]

=== get 测试 ===
Alice
Bob
Carol
None

=== 重复 key 更新 ===
True
David
[None, None, None, (23, 'David'), (33, 'Bob'), (43, 'Carol'), None, None, None, None]

=== 删除测试 ===
True
[None, None, None, (23, 'David'), <object object at 0x0000024E8F5B4590>, (43, 'Carol'), None, None, None, None]

=== 删除后继续查找后续元素 ===
Carol

=== 删除不存在的 key ===
False

=== 墓碑复用测试 ===
True
[None, None, None, (23, 'David'), (53, 'Eve'), (43, 'Carol'), None, None, None, None]
Eve

=== 环绕探测测试 ===
True
True
True
[(9, 'B'), (14, 'C'), None, None, (4, 'A')]
A
B
C

=== 哈希表填满测试 ===
True
True
True
[(0, 'A'), (1, 'B'), (2, 'C')]
继续插入：
False
[(0, 'A'), (1, 'B'), (2, 'C')]

=== 满表存在墓碑时重新插入 ===
True
[(0, 'A'), <object object at 0x0000024E8F5